In [ ]:
from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoTokenizer
import torch
from torch.utils.data import DataLoader
import numpy as np
import torch.nn as nn
import math
from tqdm import tqdm


# Tokenization
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
raw_dataset = load_dataset("sentence-transformers/codesearchnet")
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['comment', 'code'],
        num_rows: 1375067
    })
})

In [ ]:
train_test = raw_dataset["train"].train_test_split(test_size=0.05, seed=42)

validation_train = train_test["train"].train_test_split(test_size=0.05, seed=42)

full_dataset = DatasetDict({
    "train": validation_train["train"],
    "validation": validation_train["test"],
    "test": train_test["test"]
})
dataset = full_dataset.filter(lambda x: len(x['code']) > 10 and len(x['comment']) < 1000)
dataset

DatasetDict({
    train: Dataset({
        features: ['comment', 'code'],
        num_rows: 1240997
    })
    validation: Dataset({
        features: ['comment', 'code'],
        num_rows: 65316
    })
    test: Dataset({
        features: ['comment', 'code'],
        num_rows: 68754
    })
})

In [ ]:

# 1. BPE tokenizer
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# 2. Trainer with 32k tokens!
trainer = BpeTrainer(
    vocab_size=32000, # 151к -> 32к
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]", "[BOS]", "[EOS]"]
)


def batch_iterator():
    batch_size = 1000
    for i in range(0, len(raw_dataset["train"]), batch_size):
      batch = raw_dataset["train"][i : i + batch_size]
      yield batch["comment"] + batch["code"]

# 3. Training
tokenizer.train_from_iterator(batch_iterator(), trainer)


fast_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    bos_token="[BOS]",
    eos_token="[EOS]",
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]"
)

fast_tokenizer.pad_token_id = fast_tokenizer.convert_tokens_to_ids("[PAD]")
fast_tokenizer.eos_token_id = fast_tokenizer.convert_tokens_to_ids("[EOS]")
fast_tokenizer.bos_token_id = fast_tokenizer.convert_tokens_to_ids("[BOS]")
fast_tokenizer.unk_token_id = fast_tokenizer.convert_tokens_to_ids("[UNK]")
fast_tokenizer.sep_token_id = fast_tokenizer.convert_tokens_to_ids("[SEP]")
tokenizer = fast_tokenizer



In [ ]:

def tokenize_function(examples):
    concat_inputs = []
    labels = []
    max_total_length = 512

    for comment, code in zip(examples["comment"], examples["code"]):
        # 1. Format
        prompt = f"# Task: {comment}\n"
        response = f"{code}"

        # 2. Tokenization + BOS + SEP + EOS
        prompt_ids = [tokenizer.bos_token_id] + tokenizer(prompt, add_special_tokens=False)["input_ids"] + [tokenizer.sep_token_id]
        response_ids = tokenizer(response, add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]

        # Code > total
        if len(response_ids) > max_total_length:
            response_ids = response_ids[:max_total_length]
            prompt_ids = []

        # Comm + Code > total -> Code + Comm residual
        elif len(prompt_ids) + len(response_ids) > max_total_length:
            available_space = max_total_length - len(response_ids)
            # leaving only the end of the comment
            prompt_ids = prompt_ids[-available_space:]

        # 3. Comm + code
        input_ids = prompt_ids + response_ids

        # 4. -100 for prompt / comm (ignored ID)
        comment_labels = [-100] * len(prompt_ids)
        code_labels = response_ids.copy()

        label_ids = comment_labels + code_labels

        concat_inputs.append(input_ids)
        labels.append(label_ids)

    return {
        "input_ids": [ids[:512] for ids in concat_inputs],
        "labels": [lbl[:512] for lbl in labels]
    }

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=dataset["train"].column_names)

In [ ]:
class CodeLLMDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    def __call__(self, features):
        # features — это список диктов, который отдает нам Dataset:
        # [{"input_ids": [...], "labels": [...]}, {"input_ids": [...], "labels": [...]}]

        input_ids = [feature["input_ids"] for feature in features]
        labels = [feature["labels"] for feature in features]

        # 1. Dynamic pad | Searching for the longest msg in this bbatch
        max_len = max(len(ids) for ids in input_ids)

        padded_input_ids = []
        padded_labels = []

        for ids, lbls in zip(input_ids, labels):
            remainder = max_len - len(ids)

            # Adding pad_token_id
            padded_input_ids.append(ids + [self.pad_token_id] * remainder)
            # -100 (ignored id) for padding
            padded_labels.append(lbls + [-100] * remainder)

        # 2. -> Pythorch
        batch = {
            "input_ids": torch.tensor(padded_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor([
                [1 if token != self.pad_token_id else 0 for token in ids]
                for ids in padded_input_ids
            ], dtype=torch.long),
            "labels": torch.tensor(padded_labels, dtype=torch.long)
        }

        return batch

In [ ]:

data_collator = CodeLLMDataCollator(tokenizer=tokenizer)

# DataLoaders
train_dataloader = DataLoader(
    tokenized_datasets["train"],
    batch_size=64,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=2,
    pin_memory=True
)

test_dataloader = DataLoader(
    tokenized_datasets["test"],
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator
)


val_dataloader = DataLoader(
    tokenized_datasets["validation"],
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator
)

In [ ]:
# class PositionalEncoding(nn.Module):
#     def __init__(self, d_model, max_len=512):
#         super().__init__()

#         # Create a long enough 'pe' matrix with zeros
#         pe = torch.zeros(max_len, d_model)

#         # Compute the positional encodings
#         position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
#         div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

#         pe[:, 0::2] = torch.sin(position * div_term)
#         pe[:, 1::2] = torch.cos(position * div_term)

#         # Register as buffer so it is saved with the model but not trained
#         self.register_buffer('pe', pe.unsqueeze(0))

#     def forward(self, x):
#         # x shape: [batch_size, seq_len, d_model]
#         # Add positional encoding to token embeddings, slice to match sequence length
#         x = x + self.pe[:, :x.size(1)]
#         return x

In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings=2048, base=10000):
        super().__init__()
        # dim — это размер ОДНОЙ головы (d_head)
        self.dim = dim

        # Вычисляем угловые частоты (theta) по формуле из статьи
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

        # Заранее генерируем сетку позиций
        t = torch.arange(max_position_embeddings, dtype=torch.float32)

        # Внешнее произведение: получаем матрицу углов для каждой позиции [max_pos, dim // 2]
        freqss = torch.outer(t, self.inv_freq)

        # Дублируем частоты, чтобы они подходили под полный размер головы dim
        emb = torch.cat((freqss, freqss), dim=-1) # [max_pos, dim]

        cos_cached = emb.cos().unsqueeze(0).unsqueeze(1)
        sin_cached = emb.sin().unsqueeze(0).unsqueeze(1)

        # Считаем и сохраняем косинусы и синусы
        self.register_buffer("cos_cached", cos_cached, persistent=False)
        self.register_buffer("sin_cached", sin_cached, persistent=False)

    def forward(self, x, seq_len):
        # Возвращаем косинусы и синусы нужной длины, адаптированные под размеры тензора Q/K
        # Форма на выходе: [1, 1, seq_len, dim]
        return (
            self.cos_cached[:, :, :seq_len, :].to(dtype=x.dtype),
            self.sin_cached[:, :, :seq_len, :].to(dtype=x.dtype)
        )

def rotate_half(x):
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(tensor, cos, sin):
    # x * cos + rotate_half(x) * sin
    return (tensor * cos) + (rotate_half(tensor) * sin)

In [ ]:
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, n_heads_q=8, n_heads_kv=4, dropout=0.1):
    super().__init__()
    self.n_heads = n_heads_q + n_heads_kv * 2
    self.d_head = d_model // n_heads_q
    self.d_model = d_model

    self.n_heads_q = n_heads_q
    self.q_dim = n_heads_q * self.d_head
    self.kv_dim = n_heads_kv * self.d_head

    # Общий размер для одного слоя проекции
    total_dim = self.q_dim + 2 * self.kv_dim

    #
    self.qkv_proj = nn.Linear(d_model, total_dim)

    self.fc = nn.Linear(self.q_dim, d_model)
    self.dropout = nn.Dropout(dropout)
    self.rotary_emb = RotaryEmbedding(dim=self.d_head)

  def forward(self, x, padding_mask=None):
    # x [batch, padded_size, d_model]
    batch_size = x.size(0)
    seq_len = x.size(1)
    qkv = self.qkv_proj(x)
    q, k, v = torch.split(qkv, [self.q_dim, self.kv_dim, self.kv_dim], dim=-1)

    q = q.view(batch_size, seq_len, self.n_heads_q, self.d_head).transpose(1, 2)
    k = k.view(batch_size, seq_len, self.n_heads_kv, self.d_head).transpose(1, 2)
    v = v.view(batch_size, seq_len, self.n_heads_kv, self.d_head).transpose(1, 2)

    # ====== RoPE ADDED ======
    # Получаем матрицы поворота для текущей длины последовательности
    # Переносим их на устройство (GPU), где лежит входной тензор х
    cos, sin = self.rotary_emb(x, seq_len)

    q = apply_rotary_pos_emb(q, cos, sin)
    k = apply_rotary_pos_emb(k, cos, sin)
    # =======================================

    # attn_weights [batch, padded_size, padded_size]
    # torch.triu создает верхнетреугольную матрицу. Из-за diagonal=1 диагональ остается доступной.
    # Triangle matrix
    causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool, device=x.device), diagonal=1)
    causal_mask = causal_mask.unsqueeze(0).unsqueeze(1) # -> [1, 1, seq_len, seq_len]

    # 3. Adding padding mask
    if padding_mask is not None:
        while padding_mask.dim() < 4:
            # Убираем одну из единичных осей, чтобы стало [16, 1, 1, 293]
            padding_mask = padding_mask.unsqueeze(1)

        mask = torch.logical_or(causal_mask, padding_mask == 0).bool()
    else:
        mask = causal_mask

    # attn_weights = F.softmax(attn_weights, dim=-1)
    # attn_weights = self.dropout(attn_weights)
    # output = torch.bmm(attn_weights, v)
    # # output [batch, padded_size, d_head]
    # return output

    # Replaced hand calculation to more efficient scaled_dot_product_attention
    output = F.scaled_dot_product_attention(
        q, k, v,
        attn_mask=mask,
        dropout_p=self.dropout.p if self.training else 0.0,
        is_causal=False
    )
    # output -> [batch, +n_heads, seq_len, d_head]

    # output -> [batch, seq_len, d_model]
    output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.q_dim)
    # output -> [batch, seq_len, d_model]
    # Mixing heads
    output = self.fc(output)

    # output -> [batch, seq_len, d_model]
    return output # .squeeze(1)

In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self, d_model = 512, dim_feedforward=2048, dropout=0.1):
    super().__init__()

    self.head = MultiHeadAttention(d_model)

    # self.linear0 = nn.Linear(d_model, d_model)

    self.linear1 = nn.Linear(d_model, dim_feedforward)
    self.dropout = nn.Dropout(dropout)
    self.linear2 = nn.Linear(dim_feedforward, d_model)
    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.dropout1 = nn.Dropout(dropout)
    self.dropout2 = nn.Dropout(dropout)

  def forward(self, x, padding_mask):
    # x [batch, padded_size, d_model]
    head_output = self.head(x, padding_mask)
    # head_output -> [batch, padded_size, d_model]
    # used to be [batch, padded_size, d_head] each

    # 2. connecting by  (dim=-1)
    # attn_output = torch.cat(head_outputs, dim=-1)
    # attn_output -> [batch, padded_size, d_head * n_head]
    # attn_output -> [batch, padded_size, d_model]
    # attn_output = self.linear0(attn_output)

    x = x + self.dropout1(head_output)
    x = self.norm1(x)
    # [batch, padded_size, d_model] -> [batch, padded_size, dim_feedforward] -> [batch, padded_size, d_model]
    linear_output = self.linear2(self.dropout(F.relu(self.linear1(x))))
    x = x + self.dropout2(linear_output)
    x = self.norm2(x)
    # [batch, padded_size, d_model]
    return x



In [ ]:
class JuniorTransformer(nn.Module):
  def __init__(self, vocab_size, d_model=512, n_head=8, num_layers=12):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, d_model)
    # self.pos_encoder = PositionalEncoding(d_model)

    self.transformer_layers = nn.ModuleList([
            TransformerBlock(d_model=d_model, n_head=n_head) for _ in range(num_layers)
    ])
    self.final_norm = nn.LayerNorm(d_model)

    self.fc = nn.Linear(d_model, vocab_size)


  def forward(self, x, padding_mask):
    # x [batch, padded_size]

    # x -> [batch, padded_size, d_model]
    x = self.embedding(x)

    # x -> [batch, padded_size, d_model]
    #x = self.pos_encoder(x)

    for layer in self.transformer_layers:
      x = layer(x, padding_mask)

    # x = x.mean(dim=1) # x[:, 0, :]
    #x = x[:, -1, :]
    # x -> [batch, d_model]
    x = self.final_norm(x)
    x = self.fc(x)
    # x -> [batch, vocab_size]
    return x



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = JuniorTransformer(len(tokenizer))
model.to(device)

def gpt_style_init_weights(module, num_layers=12):
    if isinstance(module, nn.Linear):
        # 1. Default for all FC (Q, K, V & first layer FFN)
        std = 0.02

        # 2. УСЛОЖНЕНИЕ: Если это финальный слой в остаточной связи, уменьшаем веса
        # Мы проверяем это по имени переменной, либо можно проверять размерности.
        # В вашем TransformerBlock это слои linear0 (выход attn) и linear2 (выход ffn)
        # Если вы не уверены в именах, можно проверять их логически или передавать флаг.

        torch.nn.init.normal_(module.weight, mean=0.0, std=std)

        if module.bias is not None:
            torch.nn.init.zeros_(module.bias)

    elif isinstance(module, nn.Embedding):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    elif isinstance(module, nn.LayerNorm):
        # LN: default weight 1, bias 0
        torch.nn.init.ones_(module.weight)
        torch.nn.init.zeros_(module.bias)


model.apply(lambda m: gpt_style_init_weights(m, num_layers=12))

# Layer by layer fine-tuning weight initialization of input layers
for name, sub_module in model.named_modules():
    if "linear0" in name or "linear2" in name:
        # std for deep layers
        special_std = 0.02 / ((2 * 12) ** 0.5) # ~0.004
        with torch.no_grad():
            sub_module.weight.normal_(mean=0.0, std=special_std)
        print(f"Special init applied for: {name} (std={special_std:.4f})")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, betas=(0.9, 0.95), weight_decay=0.1)


num_epochs = 3
steps_per_epoch = len(train_dataloader)
total_steps = num_epochs * steps_per_epoch
warmup_steps = 1000
min_lr_ratio = 0.1

def lr_lambda(step):
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return min_lr_ratio + (1 - min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

criterion = nn.CrossEntropyLoss()# ignore_index=tokenizer.pad_token_id)

model = torch.compile(model)

Специальная инициализация применена для: transformer_layers.0.linear0 (std=0.0041)
Специальная инициализация применена для: transformer_layers.0.linear2 (std=0.0041)
Специальная инициализация применена для: transformer_layers.1.linear0 (std=0.0041)
Специальная инициализация применена для: transformer_layers.1.linear2 (std=0.0041)
Специальная инициализация применена для: transformer_layers.2.linear0 (std=0.0041)
Специальная инициализация применена для: transformer_layers.2.linear2 (std=0.0041)
Специальная инициализация применена для: transformer_layers.3.linear0 (std=0.0041)
Специальная инициализация применена для: transformer_layers.3.linear2 (std=0.0041)
Специальная инициализация применена для: transformer_layers.4.linear0 (std=0.0041)
Специальная инициализация применена для: transformer_layers.4.linear2 (std=0.0041)
Специальная инициализация применена для: transformer_layers.5.linear0 (std=0.0041)
Специальная инициализация применена для: transformer_layers.5.linear2 (std=0.0041)
Спец

In [ ]:
def eval(model, device, val_dataloader, criterion):
  model.eval()
  total_loss = 0.0

  pbar = tqdm(val_dataloader, desc="Evaluating", leave=False)

  with torch.no_grad():
    for batch in pbar:
      input_ids = batch['input_ids'].to(device)
      labels = batch['labels'].to(device)

      padding_mask = (input_ids[:, :-1] != tokenizer.pad_token_id).unsqueeze(1).unsqueeze(2)

      x_input = input_ids[:, :-1]
      target = labels[:, 1:]
      outputs = model(x_input, padding_mask)
      loss = criterion(outputs.reshape(-1, outputs.size(-1)), target.reshape(-1))

      total_loss += loss.item()


      pbar.set_postfix(loss=f"{loss.item():.4f}")

  return total_loss / len(val_dataloader)

def train(epoch, model, device, train_dataloader, optimizer, scheduler, criterion):
  model.train()
  optimizer.zero_grad()

  total_loss = 0.0
  save_every_steps = 5000


  pbar = tqdm(train_dataloader, desc="Training", leave=False)

  for step, batch in enumerate(pbar):
    input_ids = batch['input_ids'].to(device)
    labels = batch['labels'].to(device)

    padding_mask = (input_ids[:, :-1] != tokenizer.pad_token_id).unsqueeze(1).unsqueeze(2)


    x_input = input_ids[:, :-1]
    target = labels[:, 1:]

    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
      outputs = model(x_input, padding_mask)


      loss = criterion(outputs.reshape(-1, outputs.size(-1)), target.reshape(-1))

    loss.backward()

    # Рекомендуется: обрезка градиентов от взрыва (NaN)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    if step % 500 == 0:
      print(f"loss: {(total_loss/step):.4f} \| PPL: {torch.exp(total_loss/step):.4f}")

    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()

    total_loss += loss.item()

    pbar.set_postfix(loss=f"{loss.item():.4f}")

    # epoch step save
    if step > 0 and step % save_every_steps == 0:
        checkpoint = {
            'epoch': epoch,
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),

            'loss': loss.item(),
        }

        torch.save(checkpoint, f"checkpoint_epoch_{epoch}_step_{step}.pt")
        print(f"\n[INF] Checkpoint save {step}")

  return total_loss / len(train_dataloader)

In [ ]:
torch.set_float32_matmul_precision('high')

In [ ]:
for i in range(num_epochs):
  train_loss = train(i, model, device, train_dataloader, optimizer, scheduler, criterion)
  val_loss = eval(model, device, val_dataloader, criterion)
  checkpoint = {
            'epoch': i,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': val_loss,
        }

  torch.save(checkpoint, f"checkpoint_epoch_{i}.pt")
  print(f'Epoch {i}/10 --- Train loss: {train_loss:.4f} \| PPL {torch.exp(train_loss):.4f} --- Val loss: {val_loss:.4f} \| PPL {torch.exp(val_loss):.4f}')

Training:   0%|          | 1/19391 [03:04<995:44:49, 184.87s/it, loss=10.4307]

embedding grad_norm = 0.000e+00
fc grad_norm        = 0.000e+00


Training:   0%|          | 14/19391 [03:08<4:22:57,  1.23it/s, loss=10.3899]/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
Training:   3%|▎         | 501/19391 [10:04<1:48:14,  2.91it/s, loss=6.8687]

embedding grad_norm = 0.000e+00
fc grad_norm        = 0.000e+00


Training:   5%|▌         | 1001/19391 [12:46<1:42:03,  3.00it/s, loss=5.5976]

...

Training:  95%|█████████▌| 18501/19391 [1:47:24<04:48,  3.08it/s, loss=3.1198]

embedding grad_norm = 0.000e+00
fc grad_norm        = 0.000e+00


Training:  98%|█████████▊| 19001/19391 [1:50:06<02:07,  3.06it/s, loss=2.8149]

embedding grad_norm = 0.000e+00
fc grad_norm        = 0.000e+00


Training: 100%|█████████▉| 19390/19391 [1:52:13<00:00,  2.98it/s, loss=2.9473]/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
Evaluating:   0%|          | 0/4083 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
Evaluating:  15%|█▌        | 620/4083 [02:25<02:36, 22.10it/s, loss=2.6403]

In [ ]:
import os

# Define the path to save the model in Google Drive
save_path = "/content/drive/MyDrive/programmer_1st_epoch_checkpoint.pt"

# Save the model's state dictionary
torch.save(model.state_dict(), save_path)

print(f"Model saved successfully to: {save_path}")

Model saved successfully to: /content/drive/MyDrive/my_model_checkpoint.pt


In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=100, max_length=512,
             temperature=0.8, top_k=40):
    model.eval()

    ids = ([tokenizer.bos_token_id]
           + tokenizer(prompt, add_special_tokens=False)["input_ids"]
           + [tokenizer.sep_token_id])

    # Защита: если промпт сам по себе слишком длинный — обрезаем его конец
    if len(ids) >= max_length:
        ids = ids[-(max_length - 1):]  # оставляем место хотя бы для 1 нового токена

    x = torch.tensor([ids], device=device)
    prompt_len = x.size(1)

    for _ in range(max_new_tokens):
        # 1) Ограничиваем контекст max_length (sliding window)
        x_cond = x if x.size(1) <= max_length else x[:, -max_length:]

        mask = torch.ones(x_cond.shape, dtype=torch.bool, device=device)
        logits = model(x_cond, mask)[:, -1, :] / temperature

        # 2) top-k
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = -float('inf')
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1)

        if next_token.item() == tokenizer.eos_token_id:
            break

        x = torch.cat([x, next_token], dim=1)

        # 3) Жёсткий стоп по абсолютной длине
        if x.size(1) >= max_length:
            break

    # Декодируем только сгенерированную часть отдельно для наглядности
    generated_ids = x[0, prompt_len:].tolist()
    full_text = tokenizer.decode(x[0].tolist(), skip_special_tokens=True)
    gen_only = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return {"full": full_text, "generated": gen_only}

In [ ]:
prompts = [
    "# Task: sort a list of integers in ascending order\n",
    "# Task: read a file line by line and return list of strings\n",
    "# Task: calculate the factorial of n\n",
]

for p in prompts:
    print("=" * 60)
    print(f"PROMPT: {p}")
    result = generate(model, tokenizer, p, max_new_tokens=150)
    print(f"--- GENERATED ---\n{result['generated']}")

PROMPT: # Task: sort a list of integers in ascending order



/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(


--- GENERATED ---
def iter ( self , value = None ): """""" if value is not None : value = value . get (' value ') if value is not None : value = value . get (' value ') value = value . get (' value ') if value is not None : value = value . get (' value ') if value is not None : value = value . get (' value ') return value
PROMPT: # Task: read a file line by line and return list of strings

--- GENERATED ---
def read _file ( line ): """""" if line not in line : line = line . strip (). split ( 0 , 1 ) parts = line . split ("\ n ") if parts is not None : line = line [ 1 ] elif parts is not None : line = line [ 0 : 0 ] elif parts is not None : line = line . strip (). split ( 0 , 2 ) if len ( parts ) == 2 : line = line [ 0 : 1 ] return line
PROMPT: # Task: calculate the factorial of n

--- GENERATED ---
def calculate ( self , n , size , v ): self . n = size self . n = v self . x = v return self
